# Wstęp do optymalizacji

Optymalizacja numeryczna to dziedzina matematyki i informatyki zajmująca się poszukiwaniem najlepszego (optymalnego) rozwiązania danego problemu. Najczęściej sprowadza się to do znalezienia minimum lub maksimum pewnej funkcji celu (np. minimalizacja kosztów, maksymalizacja zysków, minimalizacja błędu modelu uczenia maszynowego).

Plan wykładu

1. Poszukiwanie optimum za pomocą naiwnego przeszukiwania (ang. brute-force) z jedną pętlą.
2. Dlaczego dodanie drugiego wymiaru (podwójna pętla) znacząco zwiększa złożoność obliczeniową?
3. Optymalizacja w praktyce, czyli zastosowanie dedykowanych algorytmów poszukujących rozwiązania.
4. Optymalizacja rozmyta, gdzie cel jest "nieostry".
5. Reprezentacja procesów chemicznych w języku Python.
6. Optymalizacja procesów chemicznych.

In [ ]:
from chempy import balance_stoichiometry
from chempy import Substance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import pyomo.environ as pyo
from pyomo.common.numeric_types import RegisterIntegerType
from sympy.core.numbers import Integer, Number, Zero, One

In [ ]:
from scipy.optimize import minimize, minimize_scalar

In [ ]:
from tqdm import tqdm

In [ ]:
RegisterIntegerType(Integer)
RegisterIntegerType(Number)
RegisterIntegerType(Zero)
RegisterIntegerType(One)

## Przeszukiwanie naiwne (brute-force): pojedyncza pętla

Funkcja do zminimalizowania

$f(x)=(x−3)^2 + 5$

Jakie będzie minimum tej funkcji ze wzoru?

In [ ]:
def func_1d(x: float) -> float:
    return (x - 3)**2 + 5

Jak komputer poradzi sobie z tym rozwiązaniem? Poszukiwanie minimum sprowadza się do "siłowego" sprawdzenia wszystkich możliwych wartości paramatru $x$. Trzeba przyjąć jednak pewne założenia początkowe względem zakresu poszukiwań.

In [ ]:
# Przestrzeń poszukiwań: od -100 do 100, z krokiem 0.001
x_values = np.arange(-100, 100, 0.001)

A następnie, po kolei, sprawdzać jak zmienia się wartość zwracana przez optymalizowaną funkcję.

In [ ]:
%%time

y_values = []
best_x = None
min_value = float("inf")

# Przeszukujemy wszystkie wartości jedna po drugiej
for x in x_values:
    y = func_1d(x)
    y_values.append(y)

    if y < min_value:
        min_value = y
        best_x = x

print(f"Znalezione minimum: {min_value:.2f} dla x = {best_x:.2f}")
print(f"Liczba sprawdzonych punktów: {len(x_values)}")

Dobrym narzędziem pomocnicznym w badaniu przebiegu funkcji może być jej wizualizacja.

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(x_values, y_values)
plt.plot(best_x, min_value, "x")

plt.title("Wykres przebiegu funkcji jednej zmiennej")
plt.xlabel("Wartość parametru funkcji")
plt.ylabel("Wynik funkcji")
plt.grid(True)
plt.show()

## Przeszukiwanie naiwne: podwójna pętla

Teraz rozpatrzmy funkcję dwóch zmiennych, np. uproszczony model kosztów zależny od ceny surowca x i czasu produkcji y:

$f(x,y)=(x−2)^2 + (y+1)^2$

Przy tych samych załozeniach co wyzej, ile wariantów mamy do sprawdzenia?

In [ ]:
len(np.arange(-100, 100, 0.001)) ** 2

Warto zmniejszyć rozmiar kroku, co dla kazdego wymiaru pozwoli zaoszczędzić 10x więcej miejsca

In [ ]:
len(np.arange(-100, 100, 0.01)) ** 2

In [ ]:
def fun_2d(x: float, y: float) -> float:
    return (x - 2)**2 + (y + 1)**2

In [ ]:
x_values_2d = np.arange(-100, 100, 0.01)
y_values_2d = np.arange(-100, 100, 0.01)

Mamy więc do czynienia z funkcją dwóch niezależnych zmiennych: $x$ i $y$. Do znalezienia optimum jedna pętla już nie wystarczy. Ponieważ obie zmienne są niezależne, dla każdej z nich musimy zastosować osobną pętlę, co pozwoli przetestować wszystkie możliwe kombinacje wartości obydwu parametrów.

Warto zauważyć, że w przypadku funkcji jednej zmiennej mieliśmy do sprawdzenia tyle wariantów parametru, ile przyjęliśmy w zakresie poszukiwań pętli. W przypadku funkcji dwóch zmiennych, mamy dwie zagnieżdżone pętle tworzące iloczyn kartezjański wartości z obydwu pętli. Zatem łączna liczba operacji będzie wynosiła kwadrat wielkości pojeczynczej pętli.

W takich sytuacjach mówimy o złożoności obliczeniowej liniowej i kwadratowej, które odpowiadają liczbie operacji do wykonania celem znalezienia optimum. Powiązane jest z tym zjawisko tzw. przekleństwa wymiarowości, gdzie w zależności od liczby zmiennych, złożoność rośnie wykładniczo względem wymiaru.

> Złożoność obliczeniowa jest formalnie wyrażana za pomocą notacji asymptotycznej, która opisuje tempo wzrostu zapotrzebowania algorytmu na zasoby wraz ze wzrostem wielkości problemu ($N$). Notacja $\mathcal{O}$ wyznacza ograniczenie górne (pesymistyczny koszt), $\Omega$ ograniczenie dolne (optymistyczny koszt), a $\Theta$ ograniczenie ścisłe, gdy zachowanie algorytmu jest identyczne w każdym przypadku. W optymalizacji metodą naiwną (brute force) dla dwóch zmiennych należy sprawdzić dokładnie każdy z $N×N$ punktów bez możliwości wcześniejszego przerwania pętli. Zatem zarówno w pesymistycznym, jak i optymistycznym scenariuszu wykonywane jest tyle samo operacji, co daje ścisłą złożoność czasową $\Theta(N^2)$. 
>
> W kontekście zasobów należy jednak rozróżnić czas od pamięci. Złożoność czasowa wynosi $Θ(N^2$), ponieważ teoretyczny czas wykonania rośnie proporcjonalnie do kwadratu liczby próbek. Natomiast złożoność pamięciowa zależy od implementacji: w przypadku iteracji przez siatkę za pomocą dwóch zagnieżdżonych pętli i rejestrowania wyłącznie dotychczasowych optimów (minimum/maksimum), zużywana jest stały obszar pamięci, co wyraża złożoność $Θ(1)$. Złożoność pamięciowa wzrosłaby do $Θ(N^2)$ tylko wtedy, gdyby algorytm najpierw wygenerował i zapisał w tablicy wszystkie punkty siatki wraz z ich wynikami przed znalezieniem najlepszego.

In [ ]:
%%time

z_values = []
best_xy = None
min_value_2d = float("inf")
licznik_iteracji = 0

for x in tqdm(x_values_2d):
    for y in y_values_2d:
        z = fun_2d(x, y)
        z_values.append(z)
        licznik_iteracji += 1

        if z < min_value_2d:
            min_value_2d = z
            best_xy = (x, y)

print(f"Znalezione minimum: {min_value_2d:.4f} dla x = {best_xy[0]:.2f}, y = {best_xy[1]:.2f}")
print(f"Liczba sprawdzonych punktów: {licznik_iteracji}")

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

X, Y = np.meshgrid(x_values_2d[::10], y_values_2d[::10])
Z = fun_2d(X, Y)

surf = ax.plot_surface(
    X,
    Y,
    Z,
    cmap="coolwarm",
    linewidth=0,
    antialiased=False,
)

ax.zaxis.set_major_formatter("{x:.02f}")

fig.colorbar(surf, shrink=0.5, aspect=5)
plt.xlabel("X")
plt.ylabel("Y")
ax.set_zlabel("Wartość funkcji")

plt.show()

## Optymalizacja w praktyce

Optymalizacja funkcji dwóch zmiennych za pomocą metody naiwnej zajęła **ponad 2 minuty** (na układzie Apple M1 Pro). Dzieje się tak z uwagi na wspomniane przekleństwo wymiarowości. Co jednak w sytuacji, gdy będziemy mieć do czynienia z funkcją trzech zmiennych (parametrów), lub nawet więcej? Współczesne (przeciętnej wielkości) sieci neuronowe są mają jednak zwykle wiele milionów parametrów...

Dlatego w praktyce stosuje się algorytmiczną optymalizację numeryczną, których celem jest przede wszystkim zmniejszenie złożoności obliczeniowej poszukiwań optymalnych rozwiązań. Autorzy tych algorytmów wielokrotnie czerpali inspiracje z procesów występujących w codziennych zjawiskach fizycznych, czy też biologicznych. W wielu przypadkach algorytmy te bazują też na czystej matematyce. Wyróżnia się tu podejścia biało-, szaro- i czarno-skrzynkowe, których działanie zależy całkowicie, częściowo lub wcale od wzoru (czy też bardziej ogólnie - wnętrza) optymalizowanej funkcji.

Jak szybko ze znalezieniem optymum funkcji dwóch zmiennych poradzi sobie dedykowany algorytm (BFGS z biblioteki SciPy)?

> BFGS (od nazwisk twórców: Broyden, Fletcher, Goldfarb, Shanno) to jeden z najpopularniejszych algorytmów optymalizacji bez ograniczeń, należący do klasy metod quasi-newtonowskich. W przeciwieństwie do klasycznej metody Newtona, która wymaga bardzo kosztownego obliczania i odwracania pełnej macierzy drugich pochodnych (Hesjanu) w każdym kroku, BFGS w sprytny sposób iteracyjnie przybliża odwrotność Hesjanu na podstawie historii zmian samego gradientu $\nabla f(x)$. Dzięki temu łączy niemal tak samo szybką (superliniową) zbieżność jak metody drugiego rzędu z relatywnie niskim kosztem obliczeniowym zbliżonym do metod pierwszego rzędu, co czyni go domyślnym wyborem w bibliotekach optymalizacji numerycznej dla gładkich funkcji o małej i średniej liczbie zmiennych.

In [ ]:
%%time

# Zaczynamy poszukiwania w przypadkowym punkcie (np. x=50, y=50)
start_coords = (50, 50)

# Algorytm optymalizacyjny z SciPy
result = minimize(lambda v: fun_2d(v[0], v[1]), start_coords, method="BFGS")

In [ ]:
print(f"Znalezione minimum: {result.fun:.2f} dla x = {result.x[0]:.2f}, y = {result.x[1]:.2f}")
print(f"Liczba sprawdzonych punktów (ewaluacji funkcji): {result.nfev}")

Zamiast milionów iteracji, algorytm potrzebował zaledwie kilkunastu/kilkudziesięciu sprawdzeń!

## Optymalizacja rozmyta

W klasycznej optymalizacji cel jest sztywny (minimalizuj $f(x, y, ...)$. W świecie rzeczywistym cele są często "rozmyte". Na przykład przy kupnie samochodu cele są dwa i nie są wyrażone w sposób ostry: cena ma być względnie niska, przebieg nie może być zbyt duży.

W optymalizacji rozmytej (opartej na logice rozmytej - Fuzzy Logic) nie mamy wartości 0 lub 1 znanych z logiki klasycznej (prawda/fałsz), ale stopień przynależności (od 0.0 do 1.0). Optymalizacja rozmyta polega na maksymalizacji stopnia zadowolenia ze wszystkich (często sprzecznych) celów równocześnie.

Poniżej prosty przykład. Celem jest znalezienie punktu pracy maszyny (zmienna temperatura - T), gdzie:
wydajność jest względnie wysoka, a koszt energii jest akceptowalnie niski.

In [ ]:
# Zmienna decyzyjna: Temperatura w stopniach Celsjusza
T = np.linspace(20, 100, 500)

In [ ]:
# 1. Funkcja przynależności dla "Wydajność jest wysoka" 
# (Wydajność rośnie wraz z temperaturą, od 40C do 80C osiąga optimum)
performance_satisfaction = np.clip((T - 40) / 40, 0, 1)

In [ ]:
# 2. Funkcja przynależności dla "Koszt jest niski"
# (Koszty energii rosną drastycznie powyżej 60C, a przy 90C są nieakceptowalne)
cost_satisfaction = np.clip((90 - T) / 30, 0, 1)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(T, performance_satisfaction, label="Cel 1: Wysoka wydajność", color="blue", linestyle="--")
plt.plot(T, cost_satisfaction, label="Cel 2: Niski koszt", color="red", linestyle="--")

plt.title("Wydajność vs. koszt")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Stopień zadowolenia (0-1)")
plt.legend()
plt.grid(True)
plt.show()

Rozwiązaniem jest zastosowanie metody Bellmana-Zadeha, która polega na agregacji przebiegu obydwu zmiennych operatorem minimum. Optymalna decyzja to punkt, w którym te dwie wartości wchodzą w najlepszy kompromis, czyli przecięcie obydwu zmiennych.

$$ \mu_D (T) = \mu_P (T) \land \mu_C (T) = \min(\mu_P (T), \mu_C (T)) $$

- T - zmienna decyzyjna: temperatura
- P - zmienna objaśniająca: zadowolenie z wydajności
- C - zmienna objaśniająca: zadowolenie z kosztu
- $\mu$ - funkcja przynależności dla zmiennych objaśniających w punkcie zmiennej decyzyjnej

In [ ]:
total_satisfaction = np.minimum(performance_satisfaction, cost_satisfaction)

Znalezienie optymalnej tempratury polega na wyznaczeniu punktu o maksymalnej wartości funkcji przynależności dla zagregowanego, funkcją minimum, przebiegu funkcji przynależności obydwu zmiennych objaśniających.

In [ ]:
opt_idx = np.argmax(total_satisfaction)
opt_T = T[opt_idx]
max_zadowolenie = total_satisfaction[opt_idx]

In [ ]:
# Wizualizacja
plt.figure(figsize=(10, 5))
plt.plot(T, performance_satisfaction, label="Cel 1: Wysoka wydajność", color="blue", linestyle="--")
plt.plot(T, cost_satisfaction, label="Cel 2: Niski koszt", color="red", linestyle="--")
plt.plot(T, total_satisfaction, label="Całkowite zadowolenie (min-max)", color="green", linewidth=2)

plt.axvline(opt_T, color="purple", linestyle=":", label=f"Optimum rozmyte: {opt_T:.1f}°C")
plt.title("Optymalizacja Rozmyta: Kompromis między wydajnością a kosztem")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Stopień zadowolenia (0-1)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print(f"Najlepszy kompromis (optimum rozmyte) osiągnięto dla Temperatury = {opt_T:.1f}°C")
print(f"Poziom zadowolenia z rozwiązania wynosi: {max_zadowolenie:.2f} (w skali 0-1)")

## Reprezentacja procesów chemicznych w języku Python

Jednym z popularnych rozwiązań do reprezentacji procesów chemicznych w języku Python jest biblioteka [chempy](https://github.com/bjodah/chempy), której instalacja odbywa się poleceniem `pip install chempy`.

Procesem, dla kilku kolejnych przykładów, będzie spalanie metanu.

Substraty:
- metan (CH4)
- tlen (O2)

Produkty:
- dwutlenek węgla (CO2)
- woda (H2O)

### Jak ustalić proporcje substratów i produktów

Biblioteka **chempy** zawiera funkcję `balance_stoichiometry`, która automatycznie bilansuje proporcje dla wskazanej reakcji.

In [ ]:
substrates = {"CH4", "O2"}
products = {"CO2", "H2O"}

In [ ]:
substrates_proportions, products_proportions = balance_stoichiometry(substrates, products)

Dla substratów powstają następujące proporcje:

In [ ]:
for item, amount in substrates_proportions.items():
    print(f"- {item} x {amount}")

Oraz dla produktów:

In [ ]:
for item, amount in products_proportions.items():
    print(f"- {item} x {amount}")

Co można wyrazić ostatecznie jako: $CH_4 + 2O_2 \to CO_2 + 2H_2O$

### Ile to wszystko waży?

Biblioteka **chempy** udostępnia także funkcjonalność wyznaczania mas bezwględnych substancji na podstawie ich wzorów chemicznych.

In [ ]:
methane = Substance.from_formula("CH4")
water = Substance.from_formula("H2O")

In [ ]:
print(f"1 mol metanu (CH4) waży: {methane.mass:.4f} gramów.")
print(f"1 mol wody (H2O) waży: {water.mass:.4f} gramów.")

## Optymalizacja procesu chemicznego

Do dyspozycji mamy butlę mieszczącą 100 moli gazu. Celem jest znalezienie takiej mieszanki metanu i tlenu, aby zmaksymalizować względny produkt spalania. 

Funkcję wyznaczającą produkt spalania można zdefiniować następująco:

In [ ]:
required_methane = substrates_proportions["CH4"]
required_oxygene = substrates_proportions["O2"]

In [ ]:
product_sum = sum(products_proportions.values())

In [ ]:
def f_opt(methane_amount: int, oxygene_amount: int) -> int:
    methane_cycles = methane_amount // required_methane
    oxygene_cyxles = oxygene_amount // required_oxygene
    real_cycles = np.minimum(methane_cycles, oxygene_cyxles)
    total_product = real_cycles * product_sum

    return total_product

### Optymalizacja naiwna

Zastosowanie optymalizacji naiwnej będzie polegało na sprawdzeniu każdej kombinacji mieszanki gazów:
- 1x metan, 99x tlen
- 2x metan, 98x tlen
- ...
- 99x metan, 1x tlen

Finalnym wynikiem będzie kombinacja, która zapewniła największy produkt spalania.

In [ ]:
%%time

total_capacity = 100

max_product = 0
optimal_methane = 0
optimal_oxygene = 0

# Pętla sprawdzająca każdą opcję (od 1 do 99 porcji metanu)
for methane_amount in range(1, total_capacity):
    # Resztę miejsca w butli wypełniamy tlenem
    oxygene_amount = total_capacity - methane_amount

    current_total_product = f_opt(methane_amount, oxygene_amount)

    # Jeśli ten wariant jest lepszy niż nasz dotychczasowy rekord, zapisujemy go
    if current_total_product > max_product:
        max_product = current_total_product
        optimal_methane = methane_amount
        optimal_oxygene = oxygene_amount

In [ ]:
print(f"Metan: {optimal_methane} moli")
print(f"Tlen: {optimal_oxygene} moli")
print(f"Mieszanka pozwoli na wyprodukowanie łącznie {max_product:.1f} moli")

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})


X, Y = np.meshgrid(np.linspace(0, 100, 50), np.linspace(0, 100, 50))
X_mod = np.where(X + Y > 100, 100 - Y, X)
Y_mod = np.where(X + Y > 100, 100 - X, Y)
Z = f_opt(X_mod, Y_mod)

surf = ax.plot_surface(
    X,
    Y,
    Z,
    cmap="coolwarm",
    linewidth=0,
    antialiased=False,
)

# ax.zaxis.set_major_locator(LinearLocator(10))
ax.zaxis.set_major_formatter("{x:.02f}")

fig.colorbar(surf, shrink=0.5, aspect=5)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Produkty spalania")

plt.show()

### Optymalizacja algorytmiczna

#### Biblioteka scipy

Biblioteka [scipy](https://scipy.org/) posiada wiele dostępnych implementacji algorytmów optymalizacji numerycznej. Istnieje tam jednak pewne ograniczenie - wartość funkcji może być tylko minimalizowana. Wymusza to dokonanie pewnych modyfikacji oryginalnej funkcji celu (optymalizowanej).

In [ ]:
def f_maximize(methane_amount: int) -> int:
    oxygene_amount = 100 - methane_amount
    methane_cycles = methane_amount // required_methane
    oxygene_cyxles = oxygene_amount // required_oxygene
    real_cycles = np.minimum(methane_cycles, oxygene_cyxles)
    total_product = real_cycles * product_sum

    return total_product

In [ ]:
def f_target(x: float) -> float:
    arg = round(float(np.asarray(x).item()))

    result_int = f_maximize(arg)

    # dla maksymalizacji, wynik musi być odwrócony
    return -float(np.asarray(result_int).item())

In [ ]:
result = minimize_scalar(f_target, bounds=(0, 100), method="bounded")

In [ ]:
print(f"Metan: {result.x.item():.01f} moli")
print(f"Tlen: {100 - result.x.item():.01f} moli")
print(f"Mieszanka pozwoli na wyprodukowanie łącznie {-result.fun.item():.1f} moli")

#### Biblioteka pyomo

Częstszym wyborem w praktyce cieszy się biblioteka [pyomo](https://www.pyomo.org/) która jest dedykowanym rozwiązaniem dla procesów optymalizacji. W przeciwieństwie do **scipy**, w bibliotece **pyomo** kod funkcji celu jest definiowany bezpośrednio za pomocą obiektów tej biblioteki. Takie podejście pozwala na sprawną i jednoznaczną definicję operacji, które będą podlegały późniejszej bezpośredniej analizie za pomocą rozwiązań automatycznych.

W pierwszej kolejności nalezy utworzyć nowy obiekt reprezentujący model matematyczny butli z gazem

In [ ]:
model = pyo.ConcreteModel()

I następnie zmienne decyzyjne

In [ ]:
model.M = pyo.Var(within=pyo.NonNegativeIntegers, bounds=(0, total_capacity))  # Ilość metanu
model.O = pyo.Var(within=pyo.NonNegativeIntegers, bounds=(0, total_capacity))  # Ilość tlenu
model.C = pyo.Var(within=pyo.NonNegativeIntegers)  # Liczba pełnych cykli reakcji

Oraz zdefiniować cel

In [ ]:
model.target = pyo.Objective(expr=model.C * float(product_sum), sense=pyo.maximize)

Pojemność butli wynosi 100 moli. Przepełnienie grozi ekslpozją, więc trzeba przyjąć ograniczenia, które pozwolą zachować właściwe proporcje mieszanki gazów.

In [ ]:
model.capacity = pyo.Constraint(expr=model.M + model.O == total_capacity)

Liczba cykli gazu nie może także przekroczyć dostępnej ilości

In [ ]:
# Zastępuje to: cycles = gas_amount // required_gas
model.methane_constraint = pyo.Constraint(expr=model.C * required_methane <= model.M)

In [ ]:
model.oxygene_constraint = pyo.Constraint(expr=model.C * required_oxygene <= model.O)

Optymalizacja

Do dokonania właściwej optymalizacji potrzebny jest **solver** czyli program lub funkcja matematyczna służąca do znajdowania rozwiązań równań, układów równań oraz problemów optymalizacyjnych. Termin solver najczęściej kojarzony jest z dodatkiem optymalizacyjnym w programie MS Excel, ale w praktyce spotyka się go częściej w formie procesu "pod spodem" rozwiązującego problemy optymalizacyjne.

Dla potrzeb poszukiwania optymalnej mieszanki gazów, zostanie wykorzystany bezpłatny solver [glpk](https://en.wikipedia.org/wiki/GNU_Linear_Programming_Kit), który musi zostać dodatkowo zainstalowany za pomocą polecenia zaleznego od środowiska:
- MacOS: `brew install glpk`
- Bash/ Google Colab: `!apt-get install -y -qq glpk-utils`
- Conda: `conda install -c conda-forge glpk`

In [ ]:
solver = pyo.SolverFactory("glpk")
result = solver.solve(model)

Wyniki, zarówno jako optymalne parametry mieszanki gazów, jak też maksymalny produkt spalania, dostępne są z poziomu utworzonego modelu matematycznego.

In [ ]:
methane_result = pyo.value(model.M)
oxygene_result = pyo.value(model.O)
cycles = pyo.value(model.C)
product_result = pyo.value(model.target)

In [ ]:
print(f"Metan: {methane_result:.01f} moli")
print(f"Tlen: {oxygene_result:.01f} moli")
print(f"Mieszanka pozwoli na wyprodukowanie łącznie {product_result:.1f} moli")

### Optymalizacja rozmyta

Celem znalezienia optymalnej mieszanki gazów także zostanie wykorzystania metoda Bellman"a-Zadeh"a. W pierwszej kolejności należy zdefiniować zmienne objaśniające, jednocześnie dodając ograniczenie w postaci maksymalnej pojemności butli.

In [ ]:
range_ = np.arange(0, total_capacity + 1)
M, O = np.meshgrid(range_, range_)

In [ ]:
V = M + O
mu_constraint = np.where(V <= total_capacity, 1.0, 0.0)

Następnie dla każdej kombinacji proporcji gazów należy wyznaczyć powstały produkt spalania.

In [ ]:
products = f_opt(M, O)

A także maksymalny produkt celem dokonania skalowania właściwego produktu do postaci wartości przynależności (0.0 - 1.0)

In [ ]:
max_products = (total_capacity // (required_methane + required_oxygene)) * product_sum

In [ ]:
mu_target = products / max_products

Znalezienie optymalnej wartości, podobnie jak w poprzednim przypadku, sprowadza się do zastosowania agregacji za pomocą funkcji $min$.

In [ ]:
mu_optimum = np.minimum(mu_target, mu_constraint)

In [ ]:
opt_idx = np.unravel_index(np.argmax(mu_optimum), mu_optimum.shape)
opt_methane = M[opt_idx]
opt_oxygene = O[opt_idx]
opt_products = products[opt_idx]

In [ ]:
print(f"Metan: {opt_methane.item()} moli")
print(f"Tlen: {opt_oxygene.item()} moli")
print(f"Mieszanka pozwoli na wyprodukowanie łącznie {opt_products} moli")

#### Wizualizacja

In [ ]:
plt.figure(figsize=(10, 8))
plt.contourf(M, O, mu_optimum.astype(float), levels=50, cmap="viridis")
plt.colorbar(label="Wartość funkcji decyzji rozmytej (\u03bc)")

x_lin = np.linspace(0, 100, 100)
y_lin = 100 - x_lin
plt.plot(x_lin, y_lin, color="red", linestyle="--", linewidth=2, label="Ograniczenie pojemności ($CH_4+O_2=100$)")

plt.scatter(opt_methane, opt_oxygene, color="red", s=100, edgecolor="black", zorder=5, 
            label=f"Optimum ({opt_methane} $CH_4$, {opt_oxygene} $O_2$)")

plt.title("Rozmyta optymalizacja spalania metanu")
plt.xlabel("Ilość metanu w butli")
plt.ylabel("Ilość tlenu w butli")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Popularne zastosowania optymalizacji w praktyce

### Łańcuch dostaw

Wyobraźmy sobie sieć handlową, która ma 3 magazyny i 50 sklepów. Celem jest dostarczenie towaru do sklepów tak, aby zminimalizować koszty transportu, ale jednocześnie nie dopuścić do braków na półkach.

1. Predykcja popytu za pomocą Uczenia Maszynowego, gdzie model analizuje dane historyczne, pogodę, dzień tygodnia i zbliżające się święta, aby wygenerować prognozę: "Sklep nr 1 w Gdańsku sprzeda jutro 120 palet wody, a sklep nr 2 w Krakowie 85 palet".
2. Zdefiniowanie zmiennych decyzyjnych - system optymalizacyjny musi wiedzieć, czym może sterować. Zmiennymi jest tutaj ilość towaru wysłana z konkretnego magazynu do konkretnego sklepu.
3. Określenie funkcji celu
   1. To "kompas" całego procesu. Funkcją celu jest matematyczne równanie opisujące całkowity koszt transportu.
   2. Koszt całkowity = (ilość z Magazyu 1 do Sklepu 1 $\times$ cena za km) + (ilość z Magazuny 1 do Sklepu 2 $\times$ cena za km) + ...
4. Nałożenie reguł ograniczeń, co jest powszechne w takich procesacg
   1. Ograniczenia podaży: Magazyn 1 ma tylko 500 palet. Suma wysyłek z tego magazynu nie może przekroczyć 500.
   2. Ograniczenia popytu: Sklep 1 musi otrzymać dokładnie 120 palet (tyle, ile przewidział model predykcyjny).
   3. Brak ujemnych wysyłek: Nie możemy wysłać np. -10 palet, czyli wszelkie zmienne muszą być $\ge 0$
5. Uruchomienie algorytmu optymalizacyjnego (solvera)

### Dynamiczne ustalanie cen

To klasyczny problem linii lotniczych, taksówek, hoteli i e-commerce. Polega na ciągłej zmianie cen w celu maksymalizacji zysku.

1. Predykcja popytu i elastyczności za pomocą Uczenia Maszynowego
   1. W tym miejscu pojawia się uczenie maszynowe (np. model regresyjny).
   2. Model analizuje dane historyczne, wyprzedzenie zakupu, dzień tygodnia, sezonowość oraz ceny konkurencji, aby oszacować funkcję popytu: "Jeśli ustalimy cenę na poziomie 1500 PLN, prawdopodobieństwo zakupu wyniesie 85%, ale przy cenie 2200 PLN spadnie do 30%".
1. Zdefiniowanie zmiennych decyzyjnych
   1. System optymalizacyjny musi wiedzieć, czym może sterować.
   2. Zmiennymi są tutaj poziomy cen biletów w poszczególnych przedziałach czasowych lub dla konkretnych pul miejsc1. .
2. Określenie funkcji celu
   1. Funkcją celu jest matematyczne równanie opisujące całkowity przychód ze sprzedaży wszystkich miejsc.
   2. Całkowity przychód = (cena w okresie 1 × oczekiwana liczba sprzedanych biletów w okresie 1) + (cena w okresie 2 × oczekiwana liczba sprzedanych biletów w okresie 2) + ...
3. Nałożenie reguł ograniczeń
   1. Ograniczenia pojemności (podaży): Samolot ma sztywną liczbę miejsc (np. 180). Łączna liczba sprzedanych biletów we wszystkich okresach nie może przekroczyć 180.
   2. Ograniczenia biznesowe i rynkowe: Cena nie może spaść poniżej progu minimalnego (np. 800 PLN, aby pokryć koszty) ani wzrosnąć powyżej limitu prawnego/wizerunkowego (np. 5000 PLN).
   3. Brak ujemnych cen i sprzedaży: Ceny oraz liczba sprzedanych miejsc muszą być nieujemne.
4. Uruchomienie algorytmu optymalizacyjnego (solvera). Solver (często programowania nieliniowego, ze względu na krzywą popytu) przelicza przestrzeń decyzyjną i wyznacza optymalną trajektorię cen w czasie, gwarantującą najwyższy możliwy zysk.

### Optymalizacja przydziału zadań lub tras (VRP - Vehicle Routing Problem)

Wyobraźmy sobie firmę kurierską, która ma 100 paczek do dostarczenia i 5 kurierów

1. Predykcja czasów przejazdu i okien dostaw za pomocą Uczenia Maszynowego
   1. W tym miejscu pojawia się uczenie maszynowe (np. model regresyjny).
   2. Model analizuje dane historyczne z GPS, natężenie ruchu, porę dnia, pogodę oraz specyfikę lokalizacji (np. dostępność miejsc parkingowych), aby wygenerować prognozę: „Przejazd z punktu A do punktu B o godzinie 14:30 zajmie 18 minut, a czas obsługi klienta pod tym adresem wyniesie 4 minuty”.
2. Zdefiniowanie zmiennych decyzyjnych
   1. System optymalizacyjny musi wiedzieć, czym może sterować (skądś to już chyba znamy...).
   2. Zmiennymi są tutaj decyzje binarne o wyborze połączeń między punktami dla konkretnych pojazdów oraz kolejność i czasy przyjazdu do poszczególnych klientów.
3. Określenie funkcji celu
   1. Funkcją celu jest matematyczne równanie opisujące całkowity koszt realizacji wszystkich tras.
   2. Koszt całkowity = (łączny przejechany dystans × koszt paliwa za km) + (łączny czas pracy kurierów × stawka godzinowa) + kary za ewentualne spóźnienia poza wyznaczone okna czasowe.
4. Nałożenie reguł ograniczeń
   1. Ograniczenia ładowności: Każda furgonetka ma limit wagi/objętości (np. maksymalnie 30 paczek lub 400 kg). Suma paczek na trasie danego kuriera nie może przekroczyć tej wartości.
   2. Ograniczenia obsługi i ciągłości tras: Każdy ze 100 klientów musi zostać odwiedzony dokładnie raz przez jednego kuriera, a trasa każdego kierowcy musi zaczynać się i kończyć w sortowni (magazynie głównym).
   3. Ograniczenia czasu pracy (okna czasowe): Czas pojedynczej trasy nie może przekroczyć 8-godzinnego dnia pracy kuriera, a przesyłki muszą dotrzeć w przedziałach godzinowych zadeklarowanych klientom.
5. Uruchomienie algorytmu optymalizacyjnego. Ze względu na gigantyczną liczbę możliwych kombinacji (problem NP-trudny), solver wykorzystuje zaawansowane metaheurystyki (np. algorytmy genetyczne, symulowane wyżarzanie, Large Neighborhood Search), aby w kilka minut znaleźć trasę bliską optymalnej, eliminując konieczność sprawdzania miliardów permutacji.